In [1]:
import sqlite3
import json
from pathlib import Path
import pandas as pd
from src.io import load_all_games

In [2]:
import importlib
import src.transforms
importlib.reload(src.transforms)
from src.transforms import make_master_df, make_turns_df

In [4]:
DB_PATH = "catan.db"
file = "C:/Users/olive/Downloads/colonist_game_2026-06-03_12-39-19.json"

In [3]:
from src.transforms import make_master_df, make_turns_df, make_avg_prog_df

In [77]:
def backfill(folder_path):
    #get list of games already in table
    conn = sqlite3.connect("catan.db")
    unique_ids = pd.read_sql("SELECT DISTINCT game_id FROM master", conn)['game_id'].tolist()
    #get list of file stems in folder
    file_stems = [f.stem for f in Path(folder_path).glob("*.json")]
    #find files that are not in db
    files_to_ingest = [stem for stem in file_stems if stem not in unique_ids]
    return files_to_ingest

In [78]:
this = backfill("data/games")
this

[]

In [5]:
def ingest_game(file):
    conn = sqlite3.connect(DB_PATH)

    with open(file, 'r') as f:
        game_data = json.load(f)
    game_id = Path(file).stem

    #insert raw json into db
    conn.execute(
        """
        INSERT OR IGNORE INTO games (game_id, raw_json)
        VALUES (?, ?)
        """,
        (game_id, json.dumps(game_data))
    )
    print(f"Inserted {game_id} JSON into games.")

    #normalize json
    df = pd.json_normalize(game_data)
    df["game_id"] = game_id
    
    #create dfs
    master = make_master_df(df)
    turns = make_turns_df(df)

    #insert master and turns into db
    master.to_sql('master', conn, if_exists='append', index=False)
    print(f"Inserted {game_id} master data into master table.")
    turns.to_sql('turns', conn, if_exists='append', index=False)
    print(f"Inserted {game_id} turns data into turns table.")

    #finish
    conn.commit()
    conn.close()
    print(f"Finished ingesting {game_id}.")
    


In [62]:
ingest_game(file)

Inserted colonist_game_2026-06-03_12-39-19 JSON into games.
Inserted colonist_game_2026-06-03_12-39-19 master data into master table.
Inserted colonist_game_2026-06-03_12-39-19 turns data into turns table.
Finished ingesting colonist_game_2026-06-03_12-39-19.


In [6]:
from src.helpers import game_cropper, count_dcs, rank_players, update_turns_df
from src.helpers import get_place_order, resource_counter, dc_counter, robber_counter

In [34]:
def make_master_df(df):
    columns = ["game_id","player", "rank", "placement_order",
           "vp_total", "vp_settle", "vp_city", "vp_dc",
           "longest_road", "largest_army", "dcs_purchased",
           "Brick", "Grain", "Ore", "Lumber", "Wool",
           "stolen_from", 'stole']

    master_df = pd.DataFrame(columns= columns)
    for _, row in df.iterrows():
        # Create list of players ranked
        game_id = row['game_id']
        players_ranked = rank_players(row)
        place_order = get_place_order(row)
        events = [event["text"] for event in row["events"]]
        for player in players_ranked:
            summ_stats = next(
                (p for p in row["playerSummary"] if p["name"] == player))
            new_row = {
                "game_id": game_id,
                "player": player,
                "rank": players_ranked.index(player) + 1,
                "placement_order": place_order.index(player) + 1,
                "vp_total": int(summ_stats['victoryPoints']),
                "vp_settle": int(summ_stats['settlements']),
                'vp_city': int(summ_stats['cities']),
                "vp_dc": int(summ_stats.get('vp_breakdown', 0) or 0),
                "longest_road": int(summ_stats.get('longest_road', 0) or 0), 
                "largest_army": int(summ_stats.get('largest_army', 0) or 0),
                "dcs_purchased": dc_counter(events, player),
                "Brick": resource_counter(events, player, 'Brick'), 
                "Grain": resource_counter(events, player, 'Grain'), 
                "Ore": resource_counter(events, player, 'Ore'), 
                "Lumber": resource_counter(events, player, 'Lumber'), 
                "Wool":resource_counter(events, player, 'Wool'),
                "stolen_from": robber_counter(events, player)[0],
                "stole": robber_counter(events, player)[1]
            }
            master_df.loc[len(master_df)] = new_row

    return master_df

In [40]:
def make_turns_df(df):
    columns = ["game_id", "timestamp", "turn", 
           "p1_vps", "p1_dcs", "p1_settles", "p1_cities",
           "p2_vps", "p2_dcs", "p2_settles", "p2_cities",
           "p3_vps", "p3_dcs", "p3_settles", "p3_cities",
           "p4_vps", "p4_dcs", "p4_settles", "p4_cities"]
    turns_df = pd.DataFrame(columns=columns)
    for _, row in df.iterrows():
        # Create list of players ranked
        game_id = row['game_id']
        timestamp= row['timestamp']
        players_ranked = rank_players(row)
        cropped_game = game_cropper(row)
        events_text = [event["text"] for event in cropped_game["events"]]
        events = cropped_game['events']
        turns = {}
        current_turn = 1
        turns[current_turn] = []
        for event in events:
            if 'won' in event['text']:
                break
            if "<hr>" in event['html']:
                current_turn += 1
                turns[current_turn] = []
            else:
                turns[current_turn].append(event['text'])
        # Remove empty turns
        turns = {turn: events for turn, events in turns.items() if events}
        #create dict for dev card info
        dc_dict = {}
        for player in players_ranked:
            dc_dict[player] = [dc_counter(events_text, player)]
        for item in row['playerSummary']:
            player = item['name']
            vps = item.get('vp_breakdown', 0)
            dc_dict[player].append(int(vps))
        new_row = {
            "game_id": game_id,
            "timestamp": timestamp,
            "turn": 0,
            "p1_vps": 2, "p1_dcs": 0, "p1_settles": 2, "p1_cities": 0,
            "p2_vps": 2, "p2_dcs": 0, "p2_settles": 2, "p2_cities": 0,
            "p3_vps": 2, "p3_dcs": 0, "p3_settles": 2, "p3_cities": 0,
            "p4_vps": 2, "p4_dcs": 0, "p4_settles": 2, "p4_cities": 0
        }
        turns_df = pd.concat([turns_df, pd.DataFrame([new_row])], ignore_index=True)
        # Create dictionary for mapping player names to dataframe columns
        player_columns = {}
        for i, player in enumerate(players_ranked, start=1):
            player_columns[player] = (f'p{i}_vps', f'p{i}_dcs', f'p{i}_settles', f'p{i}_cities')
        # Update the dataframe with the turns from game
        update_turns_df(game_id, turns_df, turns, player_columns, dc_dict)
        # Add percentage completed column
        turns_df['game_percentage'] = turns_df['turn'] / turns_df.groupby('game_id')['turn'].transform('max')
        turns_df['percentage_bin'] = (turns_df['game_percentage'] // 0.05) * 0.05
        # Correct final scores to be in 1.0 percentage bin
        turns_df.loc[turns_df['game_percentage'] == 1.0, 'percentage_bin'] = 1.0

    return turns_df  

In [7]:
#generate dataframes with existing games
raw_df = load_all_games()
master = make_master_df(raw_df)
turns = make_turns_df(raw_df)

In [15]:
master

,game_id,player,rank,placement_order,vp_total,vp_settle,vp_city,vp_dc,longest_road,largest_army,dcs_purchased,Brick,Grain,Ore,Lumber,Wool,stolen_from,stole
0,colonist_game_2026-05-10_14-16-00,MadmanMeek,1,2,10,0,6,4,0,0,6,0,25,8,15,6,0,4
1,colonist_game_2026-05-10_14-16-00,Brill3893,2,4,7,5,0,0,2,0,5,15,16,6,8,20,10,6
2,colonist_game_2026-05-10_14-16-00,Guria2501,3,1,7,4,0,1,0,2,6,0,12,6,24,9,5,6
3,colonist_game_2026-05-10_14-16-00,frostee23,4,3,5,5,0,0,0,0,2,29,12,3,0,19,4,3
4,colonist_game_2026-05-11_19-07-06,MausersPP,1,3,11,3,4,0,2,2,3,22,6,12,16,15,6,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,colonist_game_2026-06-11_20-42-33,Jhongym,4,3,5,1,4,0,0,0,3,0,8,16,3,9,0,2
188,colonist_game_2026-06-11_21-12-55,Dixie7346,1,3,10,1,8,1,0,0,2,5,24,15,6,28,4,4
189,colonist_game_2026-06-11_21-12-55,MadmanMeek,2,4,9,5,2,0,2,0,2,16,11,4,26,5,12,8
190,colonist_game_2026-06-11_21-12-55,sevensix8,3,2,8,2,4,0,0,2,7,0,6,19,1,26,6,11


In [14]:
test = turns[turns['turn']==0]
test

,game_id,timestamp,turn,p1_vps,p1_dcs,p1_settles,p1_cities,p2_vps,p2_dcs,p2_settles,...,p3_vps,p3_dcs,p3_settles,p3_cities,p4_vps,p4_dcs,p4_settles,p4_cities,game_percentage,percentage_bin
0,colonist_game_2026-05-10_14-16-00,2026-05-10_14-16-00,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
87,colonist_game_2026-05-11_19-07-06,2026-05-11_19-07-06,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
142,colonist_game_2026-05-12_14-47-19,2026-05-12_14-47-19,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
219,colonist_game_2026-05-12_15-29-18,2026-05-12_15-29-18,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
278,colonist_game_2026-05-13_12-09-57,2026-05-13_12-09-57,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
330,colonist_game_2026-05-13_12-37-54,2026-05-13_12-37-54,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
379,colonist_game_2026-05-13_17-13-13,2026-05-13_17-13-13,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
446,colonist_game_2026-05-13_17-54-44,2026-05-13_17-54-44,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
512,colonist_game_2026-05-14_19-11-00,2026-05-14_19-11-00,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0
591,colonist_game_2026-05-16_16-58-17,2026-05-16_16-58-17,0,2,0,2,0,2,0,2,...,2,0,2,0,2,0,2,0,0.0,0.0


In [16]:
conn = sqlite3.connect("catan.db")

master.to_sql(
    "master",
    conn,
    if_exists="replace",
    index=False
)

turns.to_sql(
    "turns",
    conn,
    if_exists="replace",
    index=False
)

3280

In [ ]:
#store raw json just in case
conn.execute("""
CREATE TABLE IF NOT EXISTS games (
    game_id TEXT PRIMARY KEY,
    timestamp TEXT,
    raw_json TEXT
)
""")

In [ ]:
def load_all_games(path="data/games"):
    all_games = []

    for file in Path(path).glob("*.json"):
        with open(file) as f:
            game = json.load(f)

        df = pd.json_normalize(game)
        df["game_id"] = file.stem
        all_games.append(df)

    return pd.concat(all_games, ignore_index=True)

In [13]:
for file in Path("data/games").glob("*.json"):

    with open(file) as f:
        game = json.load(f)

    conn.execute(
        """
        INSERT OR IGNORE INTO games
        (game_id, raw_json)
        VALUES (?, ?)
        """,
        (
            file.stem,
            json.dumps(game)
        )
    )

conn.commit()
conn.close()

In [15]:
conn = sqlite3.connect("catan.db")

In [46]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

,name
0,games
1,master
2,turns
